# E08: Geometrías, unión tabular e intersección

[Enunciado](../../talleres/E08.md) · [Entorno WSL](../../docs/instalacion-wsl.md)

Ejecutar en orden con el kernel del curso. Preparar las fuentes indicadas en el enunciado; no se descargan automáticamente. Las salidas se regeneran en `kit/salidas/`. Los originales se conservan.
[Linux/macOS](../../docs/instalacion-linux-macos.md). Ajustar la ruta explícita. Las operaciones se muestran directamente con pandas, GeoPandas y Rasterio; las funciones del kit quedan como referencia posterior de automatización.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

# Editar esta ruta para que coincida con el clon local.
# Linux/macOS: Path.home() / "bigdata/big-data-postgraduate"
repositorio = Path("/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate")
KIT = repositorio / "kit"
RAW = KIT / "data/raw"
OUT = KIT / "salidas" / "E08"
assert KIT.is_dir(), "Corrige repositorio antes de continuar"
OUT.mkdir(parents=True, exist_ok=True)


In [ ]:
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.plot import show
from shapely.geometry import box, shape
from IPython.display import Image


## 1. Leer y revisar CRS
Los ZIP conservan los componentes del Shapefile. No asignar un CRS para ocultar un metadato ausente. Transformar con `to_crs` cuando corresponde.

In [ ]:
municipios=gpd.read_file('zip://' + str(RAW / 'dane_municipios.zip'), engine='pyogrio')
departamentos=gpd.read_file('zip://' + str(RAW / 'dane_departamentos.zip'), engine='pyogrio')
print('CRS:',municipios.crs,departamentos.crs)
assert len(municipios)==1122
campo=next(c for c in municipios if c.lower()=='mpio_cdpmp')
municipios['codigo_municipio']=municipios[campo].astype('string').str.zfill(5)
assert municipios.codigo_municipio.is_unique
display(municipios[['codigo_municipio','geometry']].head())

## 2. Preparar una tabla municipal
Se usa AGROSAVIA con DIVIPOLA: se concilian departamento y municipio antes de agrupar. La cardinalidad territorial se comprueba y las filas sin código se reportan, sin asignarles ubicación inventada.

In [ ]:
datos=pd.read_csv(RAW / 'agrosavia.csv', dtype='string', keep_default_na=False)
dim=pd.read_csv(RAW / 'divipola.csv', dtype='string', keep_default_na=False)
dim['codigo_municipio']=dim['Código Municipio'].str.zfill(5)
# Normalizar nombres con operaciones de texto visibles, conservando originales.
for tabla_fuente, depto, municipio in [(dim, 'Nombre Departamento', 'Nombre Municipio'),
                                       (datos, 'Departamento', 'Municipio')]:
    claves = []
    for campo_nombre in [depto, municipio]:
        normalizada = (tabla_fuente[campo_nombre].str.normalize('NFKD')
            .str.encode('ascii', errors='ignore').str.decode('ascii').str.upper()
            .str.replace(r'[^A-Z0-9 ]', ' ', regex=True)
            .str.replace(r'\s+', ' ', regex=True).str.strip())
        claves.append(normalizada)
    tabla_fuente['clave'] = claves[0] + '|' + claves[1]
assert not dim['clave'].duplicated().any(), 'Revisar claves ambiguas antes del merge' 
datos=datos.merge(dim[['clave','codigo_municipio']],how='left',on='clave',validate='many_to_one')
datos['anio']=pd.to_datetime(datos['Fecha de Análisis'],format='%d/%m/%Y',errors='coerce').dt.year
print('Sin código territorial:',int(datos.codigo_municipio.isna().sum()))
print('Sin año interpretable:',int(datos.anio.isna().sum()))
anio=int(datos.anio.max())
tabla=datos.loc[datos.anio.eq(anio)].groupby('codigo_municipio').size().rename('registros').reset_index()
mapa=municipios.merge(tabla,how='left',on='codigo_municipio',validate='one_to_one',indicator=True)
assert len(mapa)==len(municipios)
print('Año:',anio,'municipios sin registros en la tabla:',int(mapa._merge.eq('left_only').sum()))
print('Códigos de la tabla sin geometría:',tabla.loc[~tabla.codigo_municipio.isin(municipios.codigo_municipio),'codigo_municipio'].tolist())
mapa.drop(columns='geometry').to_csv(OUT/'union_municipal_agrosavia.csv',index=False)

Los conteos corresponden a registros del archivo en el año elegido, no a cobertura de suelo ni a una muestra representativa. Los municipios sin correspondencia se muestran como faltantes, sin convertirlos automáticamente a cero.

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(9,8))
mapa.plot(column='registros',legend=True,missing_kwds={'color':'lightgrey'},ax=ax)
limites=ax.axis()
departamentos.to_crs(mapa.crs).boundary.plot(ax=ax,color='black',linewidth=.3)
ax.axis(limites)
ax.set_title(f'Registros AGROSAVIA, {anio}; gris: sin registros unidos')
fig.savefig(OUT/'mapa_agrosavia.png',dpi=160,bbox_inches='tight');plt.close(fig)
display(Image(filename=str(OUT/'mapa_agrosavia.png')))

## 3. Intersección y área, paso a paso
Inspeccionar geometrías vacías, ausentes e inválidas. Registrar cuántas requieren reparación antes de `make_valid`. Transformar a WGS84 para el recorte; medir hectáreas en EPSG:9377. `set_crs` no sustituye `to_crs`. Los originales no se alteran.


In [ ]:
assert municipios.crs is not None and departamentos.crs is not None
municipios_recorte = municipios[['codigo_municipio', 'geometry']].to_crs(4326).copy()
assert not (municipios_recorte.geometry.isna() | municipios_recorte.geometry.is_empty).any()
reparaciones = {'municipios': int((~municipios_recorte.geometry.is_valid).sum())}
municipios_recorte.geometry = municipios_recorte.geometry.make_valid()
productos = []
for tema in ['igac_capacidad', 'igac_quimica']:
    muestra = gpd.read_file(RAW / (tema + '.json'), engine='pyogrio')
    assert muestra.crs is not None
    assert not (muestra.geometry.isna() | muestra.geometry.is_empty).any()
    reparaciones[tema] = int((~muestra.geometry.is_valid).sum())
    muestra.geometry = muestra.geometry.make_valid()
    muestra = muestra.to_crs(municipios_recorte.crs)
    # Mantener la geometría de recorte histórica; proyectar para medir áreas.
    recorte = gpd.overlay(muestra, municipios_recorte, how='intersection', keep_geom_type=False)
    recorte['area_interseccion_ha'] = recorte.to_crs(9377).area / 10000
    recorte = recorte.loc[recorte.area_interseccion_ha > 0].copy()
    recorte['tema'] = tema
    recorte = recorte.rename(columns={'OBJECTID': 'objectid'})
    recorte['alcance'] = 'muestra_de_5_poligonos'
    productos.append(recorte)


### Escribir resultados y control
Exportar CSV, GeoJSON y GeoPackage explícitamente. Los atributos originales de área no se recalculan; `area_interseccion_ha` mide solo el recorte.


In [ ]:
resultado = gpd.GeoDataFrame(pd.concat(productos, ignore_index=True), crs=municipios_recorte.crs)
columnas = ['tema','codigo_municipio','objectid','area_interseccion_ha','alcance']
resultado[columnas].to_csv(OUT / 'intersecciones_muestra.csv', index=False)
resultado[columnas + ['geometry']].to_file(OUT / 'intersecciones_muestra.geojson', driver='GeoJSON')
resultado.to_file(OUT / 'intersecciones_muestra.gpkg', layer='intersecciones', driver='GPKG', mode='w')
control = {'municipios_leidos': len(municipios), 'intersecciones': len(resultado),
           'crs_area': 'EPSG:9377', 'geometrias_reparadas': reparaciones,
           'limitacion': 'No representa cobertura nacional ni municipal completa.'}
(OUT / 'control_geografico.json').write_text(json.dumps(control, ensure_ascii=False, indent=2))

assert len(resultado) == 15
assert resultado.area_interseccion_ha.gt(0).all()
display(control)


### Dibujar las intersecciones
Cada panel muestra una muestra de cinco polígonos. Las escalas pueden diferir; no representa cobertura municipal completa.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, (tema, color) in zip(axes, [('igac_capacidad', 'steelblue'), ('igac_quimica', 'teal')]):
    parte = resultado.loc[resultado.tema.eq(tema)]
    parte.plot(ax=ax, color=color, edgecolor='white', linewidth=.4)
    limites = ax.axis()
    municipios.loc[municipios.codigo_municipio.isin(parte.codigo_municipio)].boundary.plot(ax=ax, color='grey', linewidth=.6)
    ax.axis(limites)
    ax.set_title(tema.replace('igac_', '').replace('_', ' ').capitalize())
    ax.set_xlabel('Longitud'); ax.set_ylabel('Latitud')
fig.suptitle('IGAC: muestra de cinco polígonos por tema\nIntersecciones con municipios; WGS84; escalas distintas por panel')
fig.tight_layout(); fig.savefig(OUT / 'mapa_intersecciones.png', dpi=160); plt.close(fig)

display(Image(filename=str(OUT / 'mapa_intersecciones.png')))


## 4. Verificar el GeoPackage y los atributos
GeoPackage conserva nombres largos. Los atributos de área originales no se recalculan al recortar: compararlos con `area_interseccion_ha` solo después de revisar sus unidades. Nunca usar estas muestras como cobertura municipal completa.

In [ ]:
recuperado=gpd.read_file(OUT/'intersecciones_muestra.gpkg',layer='intersecciones')
assert len(recuperado)==len(resultado) and recuperado.crs==resultado.crs
assert set(recuperado.columns)==set(resultado.columns)
display(resultado[[c for c in resultado.columns if 'area' in c.lower()]].head())
print('GeoPackage verificado; columnas:',resultado.columns.tolist())